# Transfer Learning para Clasificación de Flores (Oxford Flowers 102)

**Dataset:** Oxford Flowers 102 (102 clases)  
**Framework:** TensorFlow + Keras  
**Objetivo:** comparar 3 enfoques:

- **Grupo A:** Entrenar **ResNet50 desde cero** (50 épocas)
- **Grupo B:** **Fine-tuning** de ResNet50 **preentrenada (ImageNet)** (20 épocas)
- **Grupo C:** **Feature extraction** (ResNet50 congelada) + **nuevo clasificador** (10 épocas)

**Métricas a comparar**
- Accuracy final (test)
- Tiempo de entrenamiento
- Épocas hasta convergencia (heurística basada en `val_accuracy`)
- Uso de memoria GPU (si hay GPU disponible)

> Nota: En la práctica, este dataset en `tensorflow_datasets` suele tener más de 6,000 imágenes. Aquí se usa la versión estándar de **TFDS** para asegurar reproducibilidad.

In [1]:
# ============================================================
# 0) Setup: imports, seeds, configuración de ejecución
# ============================================================
import os, time, platform, subprocess, json
import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Python:", platform.python_version())

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# ---- Configuración base (puedes ajustar para tu HW) ----
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CACHE_DATASET = True
PREFETCH = True

# ---- Mixed precision (recomendado con GPU moderna) ----
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy("mixed_float16")
        print("✅ Mixed precision enabled:", mixed_precision.global_policy())
    else:
        print("ℹ️ No GPU detected. Running on CPU.")
except Exception as e:
    print("⚠️ Mixed precision not enabled:", e)

# ---- GPU memory growth (si aplica) ----
try:
    gpus = tf.config.list_physical_devices('GPU')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception as e:
    print("⚠️ Could not set memory growth:", e)

/Users/patricio/Documents/github/Cienciadedatos/.venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


TensorFlow: 2.20.0
Python: 3.13.9
ℹ️ No GPU detected. Running on CPU.


## 1) Cargar dataset (TFDS)

Usaremos `tensorflow_datasets` con el dataset: **`oxford_flowers102`**.

Splits típicos:
- `train`
- `validation`
- `test`

Cada ejemplo incluye:
- `image`: imagen RGB
- `label`: clase (0..101)

In [2]:
# ============================================================
# 1) Dataset: TFDS load
# ============================================================
!pip -q install -U tensorflow-datasets

import tensorflow_datasets as tfds

DATASET_NAME = "oxford_flowers102"

(ds_train, ds_val, ds_test), ds_info = tfds.load(
    DATASET_NAME,
    split=["train", "validation", "test"],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
print("NUM_CLASSES:", NUM_CLASSES)
print(ds_info)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/Users/patricio/Documents/github/Cienciadedatos/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:  67%|██████▋   | 2/3 [00:23<00:03,  3.13s/ url]
Extraction completed...: 0 file [00:23, ? file/s]
Dl Completed...:  67%|██████▋   | 2/3 [00:23<00:11, 11.89s/ url]


KeyboardInterrupt: 

## 2) Preprocesamiento

- Resize a `224x224`
- Normalización acorde a ResNet (`tf.keras.applications.resnet.preprocess_input`)
- Data augmentation (solo train): flips, rotación ligera, zoom

> Nota: En problemas reales, augmentation suele marcar una diferencia grande cuando entrenas desde cero (Grupo A).

In [ ]:
# ============================================================
# 2) Preprocessing pipelines
# ============================================================
from tensorflow.keras.applications.resnet import preprocess_input as resnet_preprocess

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
], name="augmentation")

@tf.function
def preprocess(image, label, training=False):
    image = tf.image.resize(image, IMG_SIZE, method="bilinear")
    image = tf.cast(image, tf.float32)
    if training:
        image = data_augmentation(image, training=True)
    image = resnet_preprocess(image)  # ResNet style: scales/centers appropriately
    return image, label

def make_pipeline(ds, training=False):
    ds = ds.map(lambda x, y: preprocess(x, y, training=training),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if CACHE_DATASET:
        ds = ds.cache()
    if PREFETCH:
        ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_pipeline(ds_train, training=True)
val_ds   = make_pipeline(ds_val, training=False)
test_ds  = make_pipeline(ds_test, training=False)

# sanity check
for batch_images, batch_labels in train_ds.take(1):
    print(batch_images.shape, batch_labels.shape, batch_labels[:10].numpy())

2026-01-08 23:59:29.453216: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


## 3) Utilidades de medición

Vamos a medir:

- **Tiempo total** de `model.fit()`
- **Época de convergencia**: heurística = primera época en que `val_accuracy` deja de mejorar (patience N, min_delta d)
- **Memoria GPU**: si hay GPU, intentamos:
  - `tf.config.experimental.get_memory_info("GPU:0")` (cuando esté disponible)
  - fallback: `nvidia-smi` (si existe)

También guardaremos **accuracy final en test** para comparar modelos.

In [ ]:
# ============================================================
# 3) Measurement utilities
# ============================================================
def has_gpu():
    return len(tf.config.list_physical_devices("GPU")) > 0

def try_tf_gpu_mem():
    \"\"\"Returns dict with current/peak bytes if available, else None.\"\"\"
    if not has_gpu():
        return None
    try:
        # Available in some TF versions/builds
        info = tf.config.experimental.get_memory_info("GPU:0")
        return {"current": int(info.get("current", -1)), "peak": int(info.get("peak", -1))}
    except Exception:
        return None

def try_nvidia_smi_mem():
    \"\"\"Returns used/total MiB from nvidia-smi if available, else None.\"\"\"
    if not has_gpu():
        return None
    try:
        cmd = ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,nounits,noheader"]
        out = subprocess.check_output(cmd).decode("utf-8").strip().splitlines()
        # take first GPU
        used, total = out[0].split(",")
        return {"used_MiB": int(used.strip()), "total_MiB": int(total.strip())}
    except Exception:
        return None

class ConvergenceAndMemCallback(tf.keras.callbacks.Callback):
    def __init__(self, monitor="val_accuracy", min_delta=1e-3, patience=5):
        super().__init__()
        self.monitor = monitor
        self.min_delta = float(min_delta)
        self.patience = int(patience)
        self.best = -np.inf
        self.best_epoch = None
        self.wait = 0
        self.converged_epoch = None
        self.mem_samples = []
        self.tf_mem_peak_bytes = None
        self.nvsmi_peak_used_mib = None

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val = logs.get(self.monitor)
        if val is not None:
            if val > self.best + self.min_delta:
                self.best = val
                self.best_epoch = epoch + 1
                self.wait = 0
            else:
                self.wait += 1
                if self.converged_epoch is None and self.wait >= self.patience:
                    self.converged_epoch = epoch + 1  # first epoch we declare convergence

        # GPU memory sampling (best-effort)
        tf_mem = try_tf_gpu_mem()
        if tf_mem:
            self.tf_mem_peak_bytes = max(self.tf_mem_peak_bytes or 0, tf_mem["peak"])
            self.mem_samples.append({"epoch": epoch+1, "tf_current": tf_mem["current"], "tf_peak": tf_mem["peak"]})

        nv = try_nvidia_smi_mem()
        if nv:
            self.nvsmi_peak_used_mib = max(self.nvsmi_peak_used_mib or 0, nv["used_MiB"])
            self.mem_samples.append({"epoch": epoch+1, "nvsmi_used_MiB": nv["used_MiB"], "nvsmi_total_MiB": nv["total_MiB"]})

def bytes_to_mib(x):
    if x is None or x < 0:
        return None
    return x / (1024**2)

## 4) Construcción de modelos (ResNet50)

Usaremos **ResNet50** por ser estándar, rápida de comparar y muy estudiada.

Estrategia común para los 3 grupos:
- `include_top=False`
- `GlobalAveragePooling2D`
- `Dense(NUM_CLASSES, softmax)`

### Grupo A (desde cero)
- `weights=None`
- LR mayor al inicio, regularización recomendada

### Grupo B (fine-tuning)
- `weights="imagenet"`
- Descongelar un subconjunto de capas (por ejemplo, último bloque)
- LR bajo (e.g. 1e-5 a 1e-4)

### Grupo C (feature extraction)
- `weights="imagenet"`
- Base congelada
- Entrenar solo el clasificador (rápido, menos riesgo de overfitting)

In [ ]:
# ============================================================
# 4) Model builders
# ============================================================
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50

def build_resnet_model(num_classes, weights, trainable_base, dropout=0.2):
    base = ResNet50(
        include_top=False,
        weights=weights,
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = trainable_base

    inputs = layers.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=trainable_base)  # important for BN behavior in fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    # If using mixed precision, ensure float32 output logits for numeric stability
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)

    model = models.Model(inputs, outputs)
    return model, base

def compile_model(model, lr):
    opt = optimizers.Adam(learning_rate=lr)
    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

## 5) Función genérica de entrenamiento + evaluación

La idea es tener un runner que:
1) construya/compile el modelo,
2) entrene con callbacks,
3) mida tiempo y convergencia,
4) evalúe en test,
5) devuelva un registro para la tabla comparativa.

**Importante:** aunque fijemos 50/20/10 épocas como exige la tarea, igual mediremos un proxy de “convergencia” con la heurística para comparar.

In [ ]:
# ============================================================
# 5) Training runner
# ============================================================
def train_and_evaluate(
    group_name: str,
    build_fn,
    epochs: int,
    lr: float,
    patience_conv: int = 5,
    min_delta_conv: float = 1e-3,
    extra_callbacks=None
):
    tf.keras.backend.clear_session()

    model, base = build_fn()
    model = compile_model(model, lr=lr)

    conv_cb = ConvergenceAndMemCallback(
        monitor="val_accuracy",
        min_delta=min_delta_conv,
        patience=patience_conv
    )

    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
        ),
        conv_cb
    ]
    if extra_callbacks:
        callbacks += extra_callbacks

    # Baseline memory snapshot
    tf_mem_before = try_tf_gpu_mem()
    nv_before = try_nvidia_smi_mem()

    t0 = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    t1 = time.perf_counter()

    # Evaluation
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)

    # Memory snapshot after
    tf_mem_after = try_tf_gpu_mem()
    nv_after = try_nvidia_smi_mem()

    # Summaries
    total_time_sec = t1 - t0
    best_epoch = conv_cb.best_epoch
    converged_epoch = conv_cb.converged_epoch

    # Peak GPU memory best-effort
    tf_peak_mib = bytes_to_mib(conv_cb.tf_mem_peak_bytes) if conv_cb.tf_mem_peak_bytes else None
    nv_peak_mib = conv_cb.nvsmi_peak_used_mib if conv_cb.nvsmi_peak_used_mib else None

    record = {
        "Grupo": group_name,
        "Epocas_plan": epochs,
        "LR": lr,
        "Test_Accuracy": float(test_acc),
        "Test_Loss": float(test_loss),
        "Tiempo_total_seg": float(total_time_sec),
        "Tiempo_por_epoca_seg": float(total_time_sec / epochs),
        "Best_val_acc_epoch": best_epoch,
        "Converged_epoch": converged_epoch,
        "GPU_TF_peak_MiB": tf_peak_mib,
        "GPU_NVSMI_peak_used_MiB": nv_peak_mib,
        "GPU_TF_mem_before": tf_mem_before,
        "GPU_TF_mem_after": tf_mem_after,
        "GPU_NVSMI_mem_before": nv_before,
        "GPU_NVSMI_mem_after": nv_after,
    }
    return model, history, record, conv_cb

def plot_history(history, title=""):
    import matplotlib.pyplot as plt
    hist = history.history
    plt.figure()
    plt.plot(hist.get("accuracy", []), label="train_acc")
    plt.plot(hist.get("val_accuracy", []), label="val_acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(hist.get("loss", []), label="train_loss")
    plt.plot(hist.get("val_loss", []), label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()

## 6) Experimentos

### Grupo A — ResNet50 desde cero (50 épocas)

Recomendaciones típicas:
- LR moderado (p.ej. `1e-3` a `3e-4`) con Adam
- Más augmentation y/o regularización (dropout)
- Espera tiempos mayores y riesgo de overfitting si datos son pocos

> Si tu hardware es limitado, baja `BATCH_SIZE` o desactiva `CACHE_DATASET`.

In [ ]:
# ============================================================
# 6A) Group A: Train from scratch
# ============================================================
def build_group_A():
    model, base = build_resnet_model(NUM_CLASSES, weights=None, trainable_base=True, dropout=0.3)
    return model, base

model_A, hist_A, rec_A, cb_A = train_and_evaluate(
    group_name="A: ResNet50 desde cero",
    build_fn=build_group_A,
    epochs=50,
    lr=3e-4,
    patience_conv=6,
    min_delta_conv=1e-3
)

plot_history(hist_A, "Grupo A (desde cero)")
rec_A

### Grupo B — Fine-tuning de ResNet50 preentrenada (20 épocas)

Estrategia:
1) Cargar ResNet50 con `weights="imagenet"`
2) Congelar capas tempranas
3) Descongelar **últimos bloques** (o últimas N capas) y entrenar con LR bajo

En este notebook usaremos:
- Base preentrenada
- Descongelar **últimas ~30 capas** (ajustable)
- LR bajo `1e-5`–`1e-4`

In [ ]:
# ============================================================
# 6B) Group B: Fine-tuning pretrained
# ============================================================
def build_group_B(unfreeze_last_n=30):
    model, base = build_resnet_model(NUM_CLASSES, weights="imagenet", trainable_base=True, dropout=0.2)
    # Freeze most layers, unfreeze last N
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False
    for layer in base.layers[-unfreeze_last_n:]:
        layer.trainable = True
    return model, base

model_B, hist_B, rec_B, cb_B = train_and_evaluate(
    group_name="B: Fine-tuning ResNet50 (ImageNet)",
    build_fn=lambda: build_group_B(unfreeze_last_n=30),
    epochs=20,
    lr=1e-5,
    patience_conv=5,
    min_delta_conv=1e-3
)

plot_history(hist_B, "Grupo B (fine-tuning)")
rec_B

### Grupo C — Feature extraction + nuevo clasificador (10 épocas)

Estrategia:
- Base ResNet50 preentrenada completamente **congelada**
- Entrenar solo el “head” (clasificador)

Ventajas:
- Muy rápido
- Menos riesgo de overfitting
- Requiere menos datos y menos cómputo

Desventajas:
- Puede saturarse en performance si las features de ImageNet no capturan bien el dominio (p.ej. tareas ultra específicas).

In [ ]:
# ============================================================
# 6C) Group C: Feature extraction
# ============================================================
def build_group_C():
    model, base = build_resnet_model(NUM_CLASSES, weights="imagenet", trainable_base=False, dropout=0.2)
    return model, base

model_C, hist_C, rec_C, cb_C = train_and_evaluate(
    group_name="C: Feature extraction (ImageNet, base congelada)",
    build_fn=build_group_C,
    epochs=10,
    lr=1e-3,
    patience_conv=4,
    min_delta_conv=1e-3
)

plot_history(hist_C, "Grupo C (feature extraction)")
rec_C

## 7) Tabla comparativa

Vamos a consolidar resultados en un DataFrame.

**Notas de interpretación**
- `Best_val_acc_epoch`: época donde se logra el mejor `val_accuracy`
- `Converged_epoch`: época donde *declaramos* convergencia (patience+min_delta)
- Memoria GPU: puede ser `None` si:
  - no hay GPU,
  - TF no soporta `get_memory_info`,
  - no existe `nvidia-smi` en el entorno

In [ ]:
# ============================================================
# 7) Consolidate metrics
# ============================================================
results = pd.DataFrame([rec_A, rec_B, rec_C])
results = results.sort_values("Test_Accuracy", ascending=False)
results

## 8) Comparaciones adicionales (plots)

- Accuracy vs tiempo
- Tiempo por época
- Convergencia vs accuracy

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(results["Tiempo_total_seg"], results["Test_Accuracy"])
for _, r in results.iterrows():
    plt.text(r["Tiempo_total_seg"], r["Test_Accuracy"], r["Grupo"], fontsize=9)
plt.xlabel("Tiempo total (seg)")
plt.ylabel("Test accuracy")
plt.title("Accuracy vs Tiempo total")
plt.show()

plt.figure()
plt.bar(results["Grupo"], results["Tiempo_por_epoca_seg"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("Segundos por época")
plt.title("Tiempo por época")
plt.show()

plt.figure()
plt.bar(results["Grupo"], results["Converged_epoch"].fillna(results["Epocas_plan"]))
plt.xticks(rotation=20, ha="right")
plt.ylabel("Época (heurística de convergencia)")
plt.title("Épocas hasta convergencia (aprox.)")
plt.show()

## 9) Discusión: trade-offs y casos de uso

### Grupo A — Desde cero
**Cuándo conviene**
- Si tu dominio es muy distinto a ImageNet (p.ej., imágenes médicas de modalidad específica) **y** tienes suficiente data.
- Cuando quieres control completo y capacidad de aprender representaciones desde la distribución objetivo.

**Trade-offs**
- Mayor costo computacional y tiempo.
- Riesgo de overfitting si el dataset es pequeño/mediano.
- Suele requerir más tuning (LR schedules, augmentations, regularización).

### Grupo B — Fine-tuning
**Cuándo conviene**
- Dataset mediano, dominio relativamente cercano a ImageNet (flores, animales, objetos).
- Necesitas alto desempeño con costo razonable.
- Tienes GPU limitada pero puedes permitirte ajustar algunas capas.

**Trade-offs**
- Necesita cuidado con BatchNorm y LR bajo (para no “romper” features aprendidas).
- Puede sobreajustar si descongelas demasiado con pocos datos.

### Grupo C — Feature extraction
**Cuándo conviene**
- Baseline rápido y robusto.
- Poco tiempo/cómputo, o dataset pequeño.
- Prototipado, pruebas A/B, o edge deployment donde entrenas rápido y despliegas liviano.

**Trade-offs**
- Puede estancarse en accuracy si la tarea requiere features más específicas.
- Menor capacidad de adaptación al dominio.

---

## 10) Preguntas guía para el informe

1. ¿Qué grupo alcanzó mayor accuracy y cuál fue el costo (tiempo/memoria)?
2. ¿El patrón de convergencia fue consistente con la teoría (C converge rápido, A lento)?
3. ¿El Grupo B entrega el mejor equilibrio performance/costo?
4. ¿Qué estrategias probarías para mejorar Grupo A? (más data aug, LR schedule, label smoothing, weight decay)
5. ¿Qué estrategias probarías para mejorar Grupo C? (más capas densas, dropout, probar EfficientNet, ViT, etc.)

In [ ]:
# ============================================================
# 11) Exportar resultados
# ============================================================
out_csv = "transfer_learning_flowers102_results.csv"
results.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# También puedes guardar el DataFrame como Excel si lo deseas:
out_xlsx = "transfer_learning_flowers102_results.xlsx"
results.to_excel(out_xlsx, index=False)
print("Saved:", out_xlsx)